# Critic magnet detection: active/generic ratio → base_rate boost

**Date:** 2026-04-18.
**Hypothesis (Jake):** When BOTH top-N active critics AND generic (low-activity) critics show up in unusual numbers, the target is a "critic magnet." Remaining (unreviewed) critics should have boosted base_rates because they're more likely to pile on than their historical activity suggests.

**Why this addresses the architectural ceiling (unlike Option C):**
- Option C used `observed_count` (volume) as target-tier signal. This mis-classified late-surge targets (low observed but high actual).
- This proposal uses `observed_generic_rate` — fraction of generic critics who've shown up. That's a direct signal that "unusual reviewers are engaging," which Option C's volume-only signal missed.

**Formulation:**
```
active_set = top n_active critics by cohort activity
generic_set = everyone else
observed_generic = observed ∩ generic_set
target_z = (target_generic_rate − cohort_mean) / cohort_std

boost = 1 + k × max(0, target_z)  # only boost up

# Apply boost to remaining critics' base_rates in KDE sum:
adjusted_base_rate[c] = base_rate[c] × boost  if c is unreviewed and generic
```

**Asymmetric** (only upward boost) because the failure mode is UNDER-prediction; we want to fix that without worsening normal-rate targets.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name != 'notebooks':
    ROOT = ROOT / 'notebooks'
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

from _helpers import (
    reviews, close_date_map, gap_lookup, first_review_ts, gap_for_slug,
    combined_score_with_scores,
    build_weighted_critic_profiles, build_weighted_kde_lambda_model,
    predict_window_custom,
    critic_activity_counts,
    CACHE_DIR,
    _blended_integral, _compute_scaling_custom,
)

SHIP_ALPHA = 0.5
SHIP_SIGMA_GAP = 8.0
SHIP_N_TRAINING = 20
SHIP_BANDWIDTH_FLOOR = 0.5
SHIP_BANDWIDTH_CEIL = 0.7
SNAP_DAYS = 3

reviews_noon = reviews.copy()
day_mask = reviews_noon['timestamp_confidence'] == 'd'
reviews_noon.loc[day_mask, 'estimated_timestamp'] = (
    reviews_noon.loc[day_mask, 'estimated_timestamp'] + pd.Timedelta(hours=12)
)

activity = critic_activity_counts()
HM = ['the_drama', 'the_super_mario_galaxy_movie', 'forbidden_fruits_2026',
      'they_will_kill_you', 'you_me_and_tuscany']

CACHE = CACHE_DIR / 'critic_magnet.pkl'
print('Ready.')

## Define active/generic critic sets at each N, compute cohort baselines

In [ ]:
def get_active_generic_sets(n_active):
    sorted_critics = sorted(activity.items(), key=lambda x: -x[1])
    active = set(c for c, _ in sorted_critics[:n_active])
    generic = set(activity) - active
    return active, generic

def observed_critics_at_snap(slug):
    target_close = close_date_map[slug]
    snap_time = target_close.floor('D') - pd.Timedelta(days=SNAP_DAYS)
    obs = reviews_noon[
        (reviews_noon['movie_slug'] == slug)
        & (reviews_noon['estimated_timestamp'] < snap_time)
        & (reviews_noon['estimated_timestamp'] < target_close)
    ]
    return set(obs['reviewer_name'])

# Precompute observed critics per target
observed_map = {s: observed_critics_at_snap(s) for s in close_date_map}
print(f'Observed critic sets computed for {len(observed_map)} targets')

# For each n_active, compute cohort generic-rate distribution
cohort_baselines = {}
for n_active in [30, 50, 100]:
    active, generic = get_active_generic_sets(n_active)
    rates = []
    for slug, obs in observed_map.items():
        if slug in HM:
            continue  # don't contaminate baseline with holdout
        gen_count = len(obs & generic)
        rate = gen_count / len(generic) if len(generic) > 0 else 0
        rates.append(rate)
    rates = np.array(rates)
    cohort_baselines[n_active] = {
        'mean': float(rates.mean()),
        'std': float(rates.std()),
        'active_set': active,
        'generic_set': generic,
    }
    print(f'n_active={n_active}: generic_rate mean={rates.mean():.4f} std={rates.std():.4f}')

## Per-target z-scores

Show how the h/m targets compare to cohort baseline across different n_active choices.

In [ ]:
for n_active in [30, 50, 100]:
    cb = cohort_baselines[n_active]
    active, generic = cb['active_set'], cb['generic_set']
    print(f'\nn_active={n_active}  (cohort baseline: mean={cb["mean"]:.4f}, std={cb["std"]:.4f})')
    for slug in HM + ['lilo_and_stitch_2025', 'wicked_2024', 'a_minecraft_movie']:
        if slug not in observed_map:
            continue
        obs = observed_map[slug]
        gen_count = len(obs & generic)
        rate = gen_count / len(generic) if len(generic) > 0 else 0
        z = (rate - cb['mean']) / cb['std'] if cb['std'] > 0 else 0
        act_count = len(obs & active)
        marker = '  *** H/M' if slug in HM else ''
        print(f'  {slug:36s}  active={act_count:3d}/{n_active:3d}  generic={gen_count:4d}  rate={rate:.4f}  z={z:+.2f}{marker}')

## Boosted prediction function

In [ ]:
def boosted_predict_window(
    model, dbc_from, dbc_to, observed_critics,
    generic_set, boost_factor,
    observed_count=None, first_review_dbc=None,
    scaling_threshold=40.0, scaling_clamp=(0.5, 2.0),
):
    """predict_window with boosted base_rates for unreviewed generic critics."""
    pop_integral = model.population_prior.integrate_box_1d(dbc_to, dbc_from)
    expected = 0.0
    for _, row in model.profiles.df.iterrows():
        name = row['reviewer_name']
        if name in observed_critics:
            continue
        w = row['base_rate']
        # Apply boost only to generic critics
        if name in generic_set:
            w = w * boost_factor
        entry = model.critic_kdes.get(name)
        if entry is None:
            continue
        integral = _blended_integral(
            entry, model.population_prior, dbc_to, dbc_from, pop_integral=pop_integral,
        )
        expected += w * integral
    if observed_count is not None and first_review_dbc is not None:
        scaling = _compute_scaling_custom(
            model, dbc_from, observed_count, first_review_dbc,
            threshold=scaling_threshold, clamp=scaling_clamp,
        )
        expected *= scaling
    return expected


def target_state_and_actual(slug):
    target_close = close_date_map[slug]
    midnight_utc_dbc = (target_close - target_close.floor('D')).total_seconds() / 86400
    snap_time = target_close.floor('D') - pd.Timedelta(days=SNAP_DAYS)
    snap_dbc_eff = (target_close - snap_time).total_seconds() / 86400
    close_midnight = target_close.floor('D')

    mr_all = reviews_noon[reviews_noon['movie_slug'] == slug]
    obs = mr_all[(mr_all['estimated_timestamp'] < snap_time)
                  & (mr_all['estimated_timestamp'] < target_close)]
    if len(obs) < 3:
        return None
    state = {
        'observed_critics': set(obs['reviewer_name']),
        'observed_count': len(obs),
        'first_review_dbc': float((target_close - obs['estimated_timestamp'].min()).total_seconds() / 86400),
    }
    if state['first_review_dbc'] < snap_dbc_eff + 1.0:
        return None
    actual_p1 = int(((mr_all['estimated_timestamp'] >= snap_time) & (mr_all['estimated_timestamp'] < close_midnight)).sum())
    return {'state': state, 'midnight_utc_dbc': midnight_utc_dbc,
            'snap_dbc_eff': snap_dbc_eff, 'target_close': target_close,
            'actual': actual_p1}


def magnet_predict(slug, n_active, k_boost):
    cb = cohort_baselines[n_active]
    active, generic = cb['active_set'], cb['generic_set']

    td = target_state_and_actual(slug)
    if td is None:
        return None
    target_gap = gap_for_slug(slug)
    if target_gap is None:
        return None

    state = td['state']
    tw = state['first_review_dbc'] - td['snap_dbc_eff']

    # Compute target z-score
    obs = state['observed_critics']
    gen_count = len(obs & generic)
    target_rate = gen_count / len(generic) if len(generic) > 0 else 0
    z = (target_rate - cb['mean']) / cb['std'] if cb['std'] > 0 else 0
    boost = 1.0 + k_boost * max(0, z)

    # Build weighted KDE as usual
    scores = combined_score_with_scores(
        slug, target_gap, obs, tw,
        k=SHIP_N_TRAINING, alpha=SHIP_ALPHA, sigma_gap=SHIP_SIGMA_GAP,
    )
    if len(scores) < 5:
        return None
    try:
        profiles = build_weighted_critic_profiles(reviews_noon, close_date_map, scores, verbose=False)
        if len(profiles.df) == 0:
            return None
        model = build_weighted_kde_lambda_model(profiles, bandwidth_floor=SHIP_BANDWIDTH_FLOOR, bandwidth_ceiling=SHIP_BANDWIDTH_CEIL)
    except Exception:
        return None

    # Baseline (no boost)
    base_pred = predict_window_custom(
        model, dbc_from=td['snap_dbc_eff'], dbc_to=td['midnight_utc_dbc'],
        observed_critics=obs,
        observed_count=state['observed_count'],
        first_review_dbc=state['first_review_dbc'],
    )

    # Boosted
    boost_pred = boosted_predict_window(
        model, dbc_from=td['snap_dbc_eff'], dbc_to=td['midnight_utc_dbc'],
        observed_critics=obs,
        generic_set=generic, boost_factor=boost,
        observed_count=state['observed_count'],
        first_review_dbc=state['first_review_dbc'],
    )

    return {
        'slug': slug, 'z': z, 'boost': boost,
        'base_pred': float(base_pred), 'boost_pred': float(boost_pred),
        'actual': td['actual'],
    }

## Sweep (n_active, k_boost) and report

In [ ]:
if CACHE.exists():
    all_results = pd.read_pickle(CACHE)
    print(f'Loaded {len(all_results)} cached rows')
else:
    rows = []
    for n_active in [30, 50, 100]:
        for k_boost in [0.5, 1.0, 2.0, 5.0]:
            for slug in close_date_map:
                r = magnet_predict(slug, n_active, k_boost)
                if r is not None:
                    r['n_active'] = n_active
                    r['k_boost'] = k_boost
                    rows.append(r)
    all_results = pd.DataFrame(rows)
    all_results.to_pickle(CACHE)
    print(f'Cached {len(all_results)} rows')

all_results['err_base'] = all_results['base_pred'] - all_results['actual']
all_results['err_boost'] = all_results['boost_pred'] - all_results['actual']

# For each config, summarize
for n_active in [30, 50, 100]:
    print(f'\n=== n_active={n_active} ===')
    print(f'  {"k_boost":>8s}  {"cohort_base_MAE":>15s}  {"cohort_boost_MAE":>17s}  {"h/m_base_MAE":>13s}  {"h/m_boost_MAE":>14s}  {"h/m_base_me":>12s}  {"h/m_boost_me":>13s}')
    for k_boost in [0.5, 1.0, 2.0, 5.0]:
        sub = all_results[(all_results['n_active']==n_active) & (all_results['k_boost']==k_boost)]
        cohort = sub[~sub['slug'].isin(HM)]
        hm_sub = sub[sub['slug'].isin(HM)]
        if len(cohort) == 0:
            continue
        print(f'  {k_boost:>8.1f}  {cohort["err_base"].abs().mean():>15.2f}  {cohort["err_boost"].abs().mean():>17.2f}  {hm_sub["err_base"].abs().mean():>13.2f}  {hm_sub["err_boost"].abs().mean():>14.2f}  {hm_sub["err_base"].mean():>+12.2f}  {hm_sub["err_boost"].mean():>+13.2f}')

## Best config per-target

In [ ]:
# Pick the best (n_active, k_boost) by cohort non-regression + h/m improvement
# Heuristic: any config where boost improves h/m MAE without worsening cohort > 5%?
print('Scanning for shippable configs (cohort MAE non-worse by > 5%, h/m MAE improved):\n')
for n_active in [30, 50, 100]:
    for k_boost in [0.5, 1.0, 2.0, 5.0]:
        sub = all_results[(all_results['n_active']==n_active) & (all_results['k_boost']==k_boost)]
        cohort = sub[~sub['slug'].isin(HM)]
        hm_sub = sub[sub['slug'].isin(HM)]
        c_base = cohort['err_base'].abs().mean()
        c_boost = cohort['err_boost'].abs().mean()
        h_base = hm_sub['err_base'].abs().mean()
        h_boost = hm_sub['err_boost'].abs().mean()
        cohort_delta = (c_base - c_boost) / c_base * 100
        hm_delta = (h_base - h_boost) / h_base * 100
        if hm_delta > 0 and cohort_delta > -5:
            print(f'  n_active={n_active}, k_boost={k_boost}: cohort {cohort_delta:+.1f}% (keep), h/m {hm_delta:+.1f}%')

# Show per-target for n_active=50, k_boost=2.0 (a middle config)
for (na, kb) in [(50, 1.0), (50, 2.0), (50, 5.0), (100, 2.0)]:
    print(f'\nPer-target h/m (n_active={na}, k_boost={kb}):')
    sub = all_results[(all_results['n_active']==na) & (all_results['k_boost']==kb) & all_results['slug'].isin(HM)]
    print(f'  {"slug":32s}  {"actual":>6s}  {"z":>6s}  {"boost":>6s}  {"base":>8s}  {"boost_pr":>9s}  {"base_err":>10s}  {"boost_err":>10s}')
    for _, r in sub.iterrows():
        print(f'  {r["slug"]:32s}  {r["actual"]:>6d}  {r["z"]:+6.2f}  {r["boost"]:>6.2f}  {r["base_pred"]:>8.2f}  {r["boost_pred"]:>9.2f}  {r["err_base"]:>+10.2f}  {r["err_boost"]:>+10.2f}')

## Decision

- Shippable config = h/m MAE improves AND cohort MAE non-worse (>−5%).
- Per-target pattern: does high-z `the_drama`/`super_mario` get meaningful upward correction? Do low-z `forbidden_fruits`/`they_will_kill_you` stay untouched?
- If yes → signal validates; consider integrating into ship stack.
- If all configs either help cohort-without-h/m or h/m-without-cohort → no free win, but the mechanism is still informative.